## Kurulum (yapılandırma + içe aktarmalar)

Bu hücre temel yapılandırmayı ayarlar, kütüphaneleri içe aktarır ve küçük bir `log()` yardımcı fonksiyonu tanımlar.

* `CORPUS_DIR`, kaynak dosyalarınızın (txt/pdf/md) bulunduğu klasördür.
* `EMB_MODEL_NAME`, arama içinde bir cümle için embedding modeli seçer.
* `TOP_K`, kaç adet metin parçası getireceğimizi denetler.
* `OPENROUTER_API_KEY` environment içinde ayarlanmış olmalıdır.

In [ ]:
# === HyDE RAG — Temel Kurulum ===
import os, time, json, re
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
import faiss
import fitz  # PDF dosyaları için PyMuPDF (isteğe bağlı; yalnızca .txt/.md kullanıyorsanız atlayın)
from sentence_transformers import SentenceTransformer
import requests

def log(msg: str):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

@dataclass
class Cfg:
    CORPUS_DIR: Path = Path("data/raw_pdfs")  # Kendi klasör yolunuzla değiştirin
    EMB_MODEL_NAME: str = "sentence-transformers/all-MiniLM-L6-v2"
    TOP_K: int = 5
    MAX_CHUNK_TOKENS: int = 400  # yaklaşık kelime sayısı; basit ayırıcı
    OPENROUTER_MODEL: str = "openai/gpt-5-nano"  # İsteğinize göre değiştirin

cfg = Cfg()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    log("UYARI: OPENROUTER_API_KEY ayarlanmadı — LLM çağrılarını çalıştırmadan önce sisteminizde ayarlayın.")


## Derlem dosyalarını yükleme ve basit parçalama (chunking)

`CORPUS_DIR` üzerindeki `.txt`, `.md` ve `.pdf` uzantılı metinleri yükler.
Ardından metinleri paragraflara ve uzunluk sınırına göre küçük parçalara **ayırır** ve dosya adı ile konum gibi basit üst veriler ekler.

In [ ]:
# === Metin dosyalarını yükleme (txt/md/pdf) ve küçük parçalara ayırma ===

def read_txt(path: Path) -> str:
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

def read_pdf(path: Path) -> str:
    try:
        doc = fitz.open(path)
        parts = []
        for page in doc:
            parts.append(page.get_text("text"))
        return "\n".join(parts)
    except Exception:
        return ""

def load_corpus_texts(corpus_dir: Path) -> List[Dict[str, Any]]:
    docs = []
    for p in sorted(corpus_dir.rglob("*")):
        if p.suffix.lower() in {".txt", ".md"}:
            txt = read_txt(p)
        elif p.suffix.lower() == ".pdf":
            txt = read_pdf(p)
        else:
            continue
        if txt.strip():
            docs.append({"path": str(p), "text": txt})
    return docs

def simple_chunks(text: str, max_tokens: int = 400) -> List[str]:
    # Paragraflara ve ardından uzunluk sınırına göre hızlı ayırıcı
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks = []
    buf = []
    count = 0
    for p in paras:
        w = len(p.split())
        if count + w > max_tokens and buf:
            chunks.append("\n\n".join(buf))
            buf, count = [p], w
        else:
            buf.append(p)
            count += w
    if buf:
        chunks.append("\n\n".join(buf))
    return chunks

def build_corpus(corpus_dir: Path) -> List[Dict[str, Any]]:
    raw_docs = load_corpus_texts(corpus_dir)
    all_chunks = []
    for d in raw_docs:
        for i, ch in enumerate(simple_chunks(d["text"], cfg.MAX_CHUNK_TOKENS)):
            all_chunks.append({"doc": d["path"], "chunk_id": i, "text": ch})
    return all_chunks

log("Derlem yükleniyor ve parçalara ayrılıyor...")
CHUNKS = build_corpus(cfg.CORPUS_DIR)
log(f"Hazır: {len(set(c['doc'] for c in CHUNKS))} dosyadan {len(CHUNKS)} parça oluşturuldu.")


## Vektör Gömmeleri (Embeddings) + FAISS İndeksi

Tüm metin parçaları için cümle gömmeleri oluşturur ve bir FAISS indeksi (kosinüs benzerliği) kurar.
Vektör gömmelerini ve parça üst verilerine geriye dönük eşleme sağlayan yapıyı hafızada saklar.

In [ ]:
# === Gömmeleri ve FAISS indeksini oluşturma (kosinüs benzerliği) ===
log("Gömme modeli yükleniyor...")
emb_model = SentenceTransformer(cfg.EMB_MODEL_NAME)

def embed_texts(texts: List[str]) -> np.ndarray:
    # (N, D) boyutunda float32 döndürür
    v = emb_model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
    return v.astype("float32")

log("Derlem parçalarının vektör gömmeleri çıkarılıyor...")
CORPUS_EMB = embed_texts([c["text"] for c in CHUNKS])
dim = CORPUS_EMB.shape[1]

# Normalize ettiğimiz için iç çarpım (inner product) kosinüs benzerliğini verir
index = faiss.IndexFlatIP(dim)
index.add(CORPUS_EMB)

log(f"FAISS indeksi hazır: {index.ntotal} vektör, boyut={dim}.")


## Basit top-k Arama

Kosinüs benzerliği kullanarak bir sorgu metni için en alakalı **top-k** metin parçasını getirir.
Skor ve üst verileri içeren eşleşme listesini döndürür.

In [ ]:
# === Basit arama yardımcısı ===
def retrieve(query: str, k: int = None) -> List[Dict[str, Any]]:
    k = k or cfg.TOP_K
    qv = embed_texts([query])
    D, I = index.search(qv, k)
    hits = []
    for score, idx in zip(D[0].tolist(), I[0].tolist()):
        if idx == -1: 
            continue
        h = CHUNKS[idx].copy()
        h["score"] = float(score)
        hits.append(h)
    return hits


## LLM Yardımcısı (OpenRouter)

OpenRouter üzerinden sohbet tarzı bir LLM'i çağırmak için küçük bir sarmalayıcı (wrapper).
İsterseniz yapılandırmadaki `OPENROUTER_MODEL` değerini değiştirebilirsiniz.

In [ ]:
# === OpenRouter sohbet yardımcısı ===
def chat_llm(prompt: str, temperature: float = 0.2) -> str:
    if not OPENROUTER_API_KEY:
        raise RuntimeError("Lütfen OPENROUTER_API_KEY çevre değişkenini ayarlayın.")

    url = "https://openrouter.ai/api/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": cfg.OPENROUTER_MODEL,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt.strip()}],
    }
    r = requests.post(url, headers=headers, data=json.dumps(payload), timeout=60)
    r.raise_for_status()
    data = r.json()
    return data["choices"][0]["message"]["content"].strip()


## HyDE adımı: varsayımsal bir yanıt üretme

**HyDE Mantığı**: Önce, kullanıcının sorusuna olası bir yanıt **kurgulayın** (henüz veritabanı araması yapılmaz).
Ardından bu varsayımsal yanıtı vektöre dönüştürür ve derlemden semantik olarak daha uyumlu, güçlü kanıtlar **getirmek** için kullanırız.

Bu hücre, varsayımsal yanıt metnini elde etmek için basit bir yönlendirme (prompt) tanımlar.

In [ ]:
# === HyDE: Sorgu için varsayımsal bir yanıt üretme (atıf yapmadan) ===
HYDE_SEED_PROMPT = """Yardımsever bir uzmansın. 
Aşağıdaki kullanıcı sorusuna öz, doğru görünen ve gerçekçi bir yanıt yaz.
Harici kaynaklara değinme veya atıf yapma; sadece doğru bir yanıtın nasıl görünebileceğini yaz.
Yanıtı kısa (120-200 kelime), net ve kısa paragraflarla yapılandırılmış tut.

Soru:
{question}
"""

def hyde_generate(query: str) -> str:
    prompt = HYDE_SEED_PROMPT.format(question=query)
    return chat_llm(prompt, temperature=0.2)


## Bağlamı biçimlendirme ve nihai yanıt yönlendirmesi

Getirilen metin parçalarını `[1]..[k]` şeklinde **numaralandırılmış kesitler** olarak biçimlendirir.
Ardından modele **yalnızca** sağlanan bağlamı kullanmasını ve yanıt içinde `[i]` işaretçileriyle **atıfta bulunmasını** söyleyen nihai **yanıt yönlendirmesini** oluşturur.

In [ ]:
# === Bağlamı biçimlendirme + nihai yanıt istemi ===
def format_context(hits: List[Dict[str, Any]]) -> str:
    out = []
    for i, h in enumerate(hits, start=1):
        out.append(f"[{i}] (doküman: {Path(h['doc']).name}#{h['chunk_id']})\n{h['text']}")
    return "\n\n".join(out)

ANSWER_PROMPT = """Uzman bir asistansın. Yanıt vermek için YALNIZCA bağlam kesitlerini kullan.
Metin içinde kaynaklara [i] şeklinde atıf yap (i kesit numarasıdır).
Yanıt kesitlerde bulunmuyorsa "Bağlamda bulunamadı." de.

Soru: {question}

Bağlam:
{context}

Yanıt (Atıflar içeren 3-6 kısa madde):
"""

def llm_answer_with_citations(query: str, hits: List[Dict[str, Any]]) -> str:
    ctx = format_context(hits)
    prompt = ANSWER_PROMPT.format(question=query, context=ctx)
    return chat_llm(prompt, temperature=0.2)


## HyDE-RAG boru hattı (pipeline)

Uçtan uca **HyDE-RAG**:

1. Sorgudan varsayımsal bir yanıt metni üretin.
2. Ham sorgu yerine **varsayımsal** metni kullanarak en alakalı **top-k** parçayı getirin.
3. LLM'den atıflar ekleyerek yalnızca bu parçaları kullanarak yanıt vermesini isteyin.

In [ ]:
# === HyDE-RAG: basit boru hattı ===
def hyde_rag(query: str, k: int = None) -> Dict[str, Any]:
    k = k or cfg.TOP_K
    log("HyDE adımı: varsayımsal yanıt üretiliyor...")
    hypo = hyde_generate(query)

    log("Varsayımsal yanıt ile arama yapılıyor...")
    hits = retrieve(hypo, k=k)

    log("Atıflar içeren nihai yanıt oluşturuluyor...")
    answer = llm_answer_with_citations(query, hits)

    return {
        "query": query,
        "hypothetical": hypo,
        "hits": hits,
        "answer": answer,
    }


## Örnek kullanım

HyDE-RAG üzerinden örnek bir sorgu çalıştırın.

* Şeffaflık sağlamak için varsayımsal yanıtı yazdırır,
* Kullanılan top-k dosyaları gösterir,
* Ve atıf içeren nihai yanıtı yazdırır.

In [ ]:
# === Demostrasyon ===
QUERY = "In PINNs, how is a divergence-free constraint typically enforced in incompressible flow problems?"

res = hyde_rag(QUERY, k=cfg.TOP_K)

print("\n" + "="*80)
print("## HyDE Varsayımsal Yanıtı")
print(res["hypothetical"])

print("\n" + "="*80)
print("## Getirilen Metin Kesitleri")
for i, h in enumerate(res["hits"], start=1):
    print(f"[{i}] {Path(h['doc']).name}#{h['chunk_id']}  (skor={h['score']:.3f})")

print("\n" + "="*80)
print("## Nihai Yanıt (Atıflı)")
print(res["answer"])
